In [ ]:
# PARAMETERS — edit this cell only.
from __future__ import annotations

import json
import os
import re
import sys
from pathlib import Path


def locate_bundle() -> Path:
    starts = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for candidate in starts:
        if (candidate / "data" / "example_tiles.csv").is_file():
            return candidate
        nested = candidate / "tasks" / "260731_2_mllm_notebook_bundle"
        if (nested / "data" / "example_tiles.csv").is_file():
            return nested
    return Path.cwd().resolve()


BUNDLE_DIR = locate_bundle()
PROJECT_DIR = next(
    (candidate for candidate in [BUNDLE_DIR, *BUNDLE_DIR.parents]
     if (candidate / "experiments").is_dir() and (candidate / "scripts").is_dir()),
    BUNDLE_DIR,
)
sys.path.append(str(PROJECT_DIR))

# Required for live tile loading. Leave empty for the included six-tile examples.
COUNTING_DATA_ROOT = os.environ.get("COUNTING_DATA_ROOT", "")
TASKS = [
    "malaria/trophozoite", "malaria/difficult", "malaria/red_blood_cell",
    "birds/bird", "dota/tennis_court", "dota/harbor", "dota/ship",
    "dota/plane", "dota/small_vehicle", "kenyan_wildlife/giraffe",
    "dota/helicopter", "dota/storage_tank",
]
MODEL_LABEL = "Qwen3-VL-8B-Instruct"
BATCH_SIZE = 8
MAX_CHOICE = 300
GPU_MEMORY_UTILIZATION = 0.85
LABEL_ROOT = PROJECT_DIR / "outputs" / "mllm_all_tiles_all_models"
PROPRIETARY_ROOT = PROJECT_DIR / "outputs" / "proprietary_all_tiles_260710"
RUN_ROOT = PROJECT_DIR / "outputs" / "runs"
EXAMPLE_CSV = BUNDLE_DIR / "data" / "example_tiles.csv"
OPEN_SOURCE_CSV = PROJECT_DIR / "outputs" / "runs" / "open_source_all_tiles_current.csv" if (PROJECT_DIR / "outputs" / "runs" / "open_source_all_tiles_current.csv").is_file() else BUNDLE_DIR / "data" / "open_source_reference_sample.csv"
PROPRIETARY_CSV = PROJECT_DIR / "outputs" / "runs" / "proprietary_all_tiles_current.csv" if (PROJECT_DIR / "outputs" / "runs" / "proprietary_all_tiles_current.csv").is_file() else BUNDLE_DIR / "data" / "proprietary_reference_sample.csv"
OPENROUTER_KEY_SELECTOR = "OPENROUTER_API_KEY"

try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except NameError:
    pass

# Safety default: inspect existing cache files first. Set True at the top for a live run.
RUN_INFERENCE = False
RECORD_TIMING = True


# 01 — Open-source MLLM inference on one GPU

This is the all-tiles inference stage as one notebook. It keeps the two-pass count → integer-choice protocol and the resumable JSONL cache, but replaces command-line dispatch with the parameters above. `RUN_INFERENCE=False` is intentional: it makes the notebook safe to inspect before downloading a model. Set it to `True` after choosing one model and confirming `COUNTING_DATA_ROOT`.


In [ ]:
import time
from random import shuffle

import pandas as pd
from IPython.display import display

MODELS = {
    "Qwen3-VL-8B-Instruct": {"modelstr": "Qwen/Qwen3-VL-8B-Instruct", "engine": "vllm", "prompt_style": "chat"},
    "InternVL3.5-8B": {"modelstr": "OpenGVLab/InternVL3_5-8B-HF", "engine": "transformers", "prompt_style": "chat"},
    "Molmo-7B-D-0924": {"modelstr": "allenai/Molmo-7B-D-0924", "engine": "vllm", "prompt_style": "molmo"},
    "Molmo2-8B": {"modelstr": "allenai/Molmo2-8B", "engine": "vllm", "prompt_style": "chat"},
    "Aya-Vision-8B": {"modelstr": "CohereLabs/aya-vision-8b", "engine": "vllm", "prompt_style": "chat"},
}
if MODEL_LABEL not in MODELS:
    raise ValueError(f"Choose MODEL_LABEL from {sorted(MODELS)}")
MODEL_CFG = MODELS[MODEL_LABEL]


def object_name(class_key: str) -> str:
    return class_key.split("/", 1)[1].replace("_", " ")


def prompt_for(name: str) -> str:
    return f"How many {name} are in this image? Only output the number and no other text."


def parse_count(text: str) -> int:
    match = re.search(r"\d+", text)
    return int(match.group(0)) if match else -1


def cache_path(class_key: str) -> Path:
    dataset, task = class_key.split("/", 1)
    return LABEL_ROOT / MODEL_LABEL / f"{dataset}__{task}.jsonl"


def read_seen_ids(path: Path) -> set[str]:
    if not path.is_file():
        return set()
    return {json.loads(line)["id"] for line in path.read_text().splitlines() if line.strip()}


def append_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, sort_keys=True) + "\n")


def require_counting_data_root() -> Path:
    if not COUNTING_DATA_ROOT:
        raise RuntimeError("Set COUNTING_DATA_ROOT before RUN_INFERENCE=True; it must contain raw/ and counting/data/.")
    root = Path(COUNTING_DATA_ROOT).expanduser().resolve()
    if not (root / "raw").is_dir() or not (root / "counting" / "data").is_dir():
        raise FileNotFoundError(f"COUNTING_DATA_ROOT must contain raw/ and counting/data/: {root}")
    return root


def load_tile_helpers():
    import torch
    from argparse import Namespace
    from torchvision.transforms.functional import to_pil_image
    from experiments.run import build_dataset

    data_root = require_counting_data_root()
    tile_sizes = PROJECT_DIR / "outputs" / "suggested_tile_sizes.json"
    image_pool_dir = PROJECT_DIR / "data" / "image_pools"

    def make_dataset(class_key: str):
        dataset, task = class_key.split("/", 1)
        args = Namespace(
            index_root=str(data_root / "counting" / "data"), raw_root=str(data_root / "raw"),
            class_key=class_key, tile_sizes=str(tile_sizes), tile_size=None,
            max_whole_image_side=1536, tile_threshold=None, pool_size=None,
            pool_sample_seed=0, image_pool_list=str(image_pool_dir / f"{dataset}__{task}__budget5000_seed0.txt"),
        )
        return build_dataset(args)

    def rows_for(dataset):
        return [{"class_key": dataset.class_key, "object_name": object_name(dataset.class_key), **row} for row in dataset.manifest()]

    def image_for(dataset, tile_index: int):
        image_tensor, _target, _meta = dataset[tile_index]
        return to_pil_image(image_tensor.clamp(0, 1))

    return torch, make_dataset, rows_for, image_for


## Load the engine only when a live run is requested

The two model paths mirror the source: vLLM handles the batched multimodal models; the Transformers fallback handles models that do not expose the same vLLM interface. Imports are delayed so cache inspection and plotting do not allocate GPU memory.


In [ ]:
def build_live_engine():
    torch, make_dataset, rows_for, image_for = load_tile_helpers()
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available. Run this notebook on a node with one visible GPU.")
    cfg = MODEL_CFG
    if cfg["engine"] == "vllm":
        from vllm import LLM
        model = LLM(
            model=cfg["modelstr"], enforce_eager=True, trust_remote_code=True,
            dtype="bfloat16", limit_mm_per_prompt={"image": 1},
            gpu_memory_utilization=GPU_MEMORY_UTILIZATION, max_num_batched_tokens=32768,
        )
        return torch, model, model.get_tokenizer(), make_dataset, rows_for, image_for
    from transformers import AutoModelForImageTextToText, AutoProcessor
    processor = AutoProcessor.from_pretrained(cfg["modelstr"], trust_remote_code=True)
    model = AutoModelForImageTextToText.from_pretrained(
        cfg["modelstr"], trust_remote_code=True, torch_dtype=torch.bfloat16, device_map="cuda"
    )
    model.eval()
    return torch, model, processor, make_dataset, rows_for, image_for


def chat_prompt(processor, text: str, with_image: bool) -> str:
    if MODEL_CFG["prompt_style"] == "molmo":
        return f"User: <image> {text} Assistant:"
    content = ([{"type": "image"}] if with_image else []) + [{"type": "text", "text": text}]
    return processor.apply_chat_template([{"role": "user", "content": content}], add_generation_prompt=True, tokenize=False)


def run_vllm_batch(model, processor, dataset, batch: list[dict], image_for) -> list[dict]:
    from vllm import SamplingParams
    count_requests = [{"prompt": chat_prompt(processor, prompt_for(row["object_name"]), True), "multi_modal_data": {"image": image_for(dataset, row["tile_index"])}} for row in batch]
    count_outputs = model.generate(count_requests, sampling_params=SamplingParams(max_tokens=512, temperature=0), use_tqdm=False)
    nlg_texts = [item.outputs[0].text.strip() for item in count_outputs]
    choice_requests = [{"prompt": chat_prompt(processor, "What number was this response trying to say?\n\nResponse:\n" + text + f"\n\nAnswer only with one integer from 0 to {MAX_CHOICE}.", False)} for text in nlg_texts]
    choice_outputs = model.generate(choice_requests, sampling_params=SamplingParams(max_tokens=8, temperature=0), use_tqdm=False)
    return [{**row, "modelstr": MODEL_CFG["modelstr"], "model_label": MODEL_LABEL, "response": {"nlg": nlg, "nlg2choice": choice.outputs[0].text.strip()}, "answer": parse_count(choice.outputs[0].text)} for row, nlg, choice in zip(batch, nlg_texts, choice_outputs)]


def run_transformers_one(model, processor, image, text: str, max_new_tokens: int) -> str:
    import torch
    prompt = chat_prompt(processor, text, image is not None)
    inputs = processor(text=prompt, images=image, return_tensors="pt") if image is not None else processor(text=prompt, return_tensors="pt")
    inputs = {key: value.to("cuda") if hasattr(value, "to") else value for key, value in inputs.items()}
    with torch.inference_mode():
        output_ids = model.generate(**inputs, do_sample=False, max_new_tokens=max_new_tokens)
    input_len = inputs["input_ids"].shape[-1]
    return processor.batch_decode(output_ids[:, input_len:], skip_special_tokens=True)[0].strip()


def run_transformers_batch(model, processor, dataset, batch: list[dict], image_for) -> list[dict]:
    rows = []
    for row in batch:
        nlg = run_transformers_one(model, processor, image_for(dataset, row["tile_index"]), prompt_for(row["object_name"]), 512)
        choice = run_transformers_one(model, processor, None, f"What number was this response trying to say?\n\nResponse:\n{nlg}\n\nAnswer only with one integer from 0 to {MAX_CHOICE}.", 8)
        rows.append({**row, "modelstr": MODEL_CFG["modelstr"], "model_label": MODEL_LABEL, "response": {"nlg": nlg, "nlg2choice": choice}, "answer": parse_count(choice)})
    return rows


## Run, resume, and record direct timing

Existing IDs are skipped, so interrupting a run does not duplicate labels. Timing is measured around this direct notebook call and is written only if `RECORD_TIMING=True`; the timing notebook consumes those records later.


In [ ]:
if RUN_INFERENCE:
    torch, model, processor, make_dataset, rows_for, image_for = build_live_engine()
    timing_path = LABEL_ROOT.parent / "mllm_notebook_timing.jsonl"
    for class_key in TASKS:
        dataset = make_dataset(class_key)
        rows = rows_for(dataset)
        for row in rows:
            row["id"] = f"class_key:{class_key};tile_index:{row['tile_index']};modelstr:{MODEL_CFG['modelstr']}"
        path = cache_path(class_key)
        seen = read_seen_ids(path)
        todo = [row for row in rows if row["id"] not in seen]
        shuffle(todo)
        started = time.perf_counter()
        written = 0
        for begin in range(0, len(todo), BATCH_SIZE):
            batch = todo[begin:begin + BATCH_SIZE]
            results = run_vllm_batch(model, processor, dataset, batch, image_for) if MODEL_CFG["engine"] == "vllm" else run_transformers_batch(model, processor, dataset, batch, image_for)
            append_jsonl(path, results)
            written += len(results)
        elapsed = time.perf_counter() - started
        if RECORD_TIMING:
            append_jsonl(timing_path, [{"model_label": MODEL_LABEL, "modelstr": MODEL_CFG["modelstr"], "class_key": class_key, "labels": len(todo), "wall_clock_sec": round(elapsed, 3), "source": "direct_notebook_wall_clock"}])
        print({"class_key": class_key, "written": written, "total": len(rows), "cache": str(path), "elapsed_sec": round(elapsed, 2)})
    del model
    torch.cuda.empty_cache()
else:
    print("Preview mode: no model loaded and no files written. Set RUN_INFERENCE=True in the parameter cell for a live run.")


## Visual sanity check: prediction versus truth and a count distribution

This view uses the included compact CSV by default, so it is available immediately. It is the quickest way to catch a shifted tile index, a bad parse, or a model that returns the same answer for every image.


In [ ]:
import pandas as pd
from IPython.display import display
from mu.visv2.figure import Figure

if OPEN_SOURCE_CSV.is_file():
    cached = pd.read_csv(OPEN_SOURCE_CSV)
    cached = cached[cached["model_label"] == MODEL_LABEL].copy()
    if cached.empty:
        cached = pd.read_csv(OPEN_SOURCE_CSV)
    task_for_plot = TASKS[0]
    task_rows = cached[(cached["dataset"] + "/" + cached["task"]) == task_for_plot]
    fig = Figure(n_rows=1, n_cols=2, figsize=(12, 4.5), fontfamily="sans-serif", fontsize=11, panel_padding=(0.28, 0.2))
    left, right = fig.panels
    left.p.scatter(task_rows["ground_truth_label"], task_rows["final_value"], s=10, alpha=0.28, color="#2563eb")
    maximum = max(task_rows["ground_truth_label"].max(), task_rows["final_value"].max())
    left.p.plot([0, maximum], [0, maximum], color="#dc2626", linestyle="--")
    left.set_xaxislabel("true count")
    left.set_yaxislabel("predicted count")
    left.set_title(f"{MODEL_LABEL}: {task_for_plot}")
    right.p.hist(cached["final_value"], bins=30, alpha=0.75, color="#0f766e", label="prediction")
    right.p.hist(cached["ground_truth_label"], bins=30, alpha=0.35, color="#f59e0b", label="truth")
    right.set_xaxislabel("count")
    right.set_yaxislabel("tiles")
    right.set_title("Count distribution")
    right.p.legend(frameon=False)
    fig.show()
else:
    print(f"Missing {OPEN_SOURCE_CSV}; run the export notebook first or set OPEN_SOURCE_CSV in the parameter cell.")
